# Conditional Strategies II: Timing Expected Returns

The ideas are in the slides. Here you test the most famous timing signal, the dividend yield: in sample, then the honest
way, out of sample. Then you try 24 timing signals on the six factors and see how many survive.

**The plan**

1. The dividend yield
2. 🎯 The forecasting regression at three horizons
3. 🎯 Out of sample: your turn
4. The timing strategy
5. Timing the six factors: 24 tries. Then back to the slides
6. 🎯 Challenge (homework): one number and three sentences

📑 **Slides for this class:** [Conditional Strategies II (PDF)](https://amoreira2.github.io/UG54/slides/conditional_strategies_2.pdf)

<iframe src="../../slides/conditional_strategies_2.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

crsp = pd.read_csv(f"{BASE}/Markettiming_data1.csv", parse_dates=['date'], index_col='date')   # CRSP market, with dividends
crsp.index = crsp.index + pd.offsets.MonthEnd(0)
rf = pd.read_parquet(f"{BASE}/factors_monthly_1926.parquet")['RF']                              # T-bill rate
ff = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)                       # the six factors, 1980-2026
F  = ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']].dropna()

C = crsp.join(rf, how='inner').dropna()
C['ex'] = C['vwretd'] - C['RF']                             # the market's excess return
C['dp'] = C['dp'].rolling(12).sum()                         # dividend yield: last 12 months of dividends over today's price
C['lx'] = np.log1p(C['vwretd']) - np.log1p(C['RF'])         # log excess return: these add up across months

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)

print(f"market {C.index[0]:%Y-%m} to {C.index[-1]:%Y-%m}; six factors {F.index[0]:%Y-%m} to {F.index[-1]:%Y-%m}")

---

## 1. The dividend yield

When prices are low relative to dividends, either dividends are about to fall, or future returns are high. Compare the
dividend yield with the return over the *next* five years.

In [ ]:
next5 = C['lx'].rolling(60).sum().shift(-60) / 5            # return over the next 60 months, a year

ax = C['dp'].plot(label='dividend yield', legend=True)
next5.plot(ax=ax, secondary_y=True, label='return over the next five years, a year', legend=True)
plt.show()

print(f"correlation: {C['dp'].corr(next5):.2f}")

- When was the dividend yield highest? What came next?
- The yield fell from about 4% in the 1980s to about 1% by 2000. Name two reasons that could happen.

---

## 2. 🎯 The forecasting regression at three horizons

$$r_{t \to t+h} = a + b \cdot \text{D/P}_t + u$$

The same regression, run for the next month, the next year and the next five years. Two sets of $t$-statistics: the naive
ones, and HAC ones that allow for the overlap.

In [ ]:
rows = {}
for h in [1, 12, 60]:
    y = C['lx'].rolling(h).sum().shift(-h) * 12 / h        # return over the next h months, a year
    data = pd.DataFrame({'y': y, 'dp': C['dp']}).loc['1927':].dropna()
    X = sm.add_constant(data['dp'])

    naive = sm.OLS(data['y'], X).fit()                                                  # as if every month were new
    hac   = sm.OLS(data['y'], X).fit(cov_type='HAC', cov_kwds={'maxlags': max(h, 1)})  # allows for the overlap

    rows[f"{h} months"] = {'b': naive.params['dp'], 'naive t': naive.tvalues['dp'], 'HAC t': hac.tvalues['dp'],
                           'R2': naive.rsquared, 'observations': len(data), 'do not overlap': len(data) // h}
print(pd.DataFrame(rows).T.round(3).to_string())

- The $R^2$ and the naive $t$ both grow with the horizon. Should you believe the five-year regression most?
- Two consecutive five-year returns share 59 of their 60 months. How many truly separate five-year periods are there?
- Which $t$-statistic would you report?

---

## 3. 🎯 Out of sample

The honest test: stand in each month $t$, using only what you knew then.

- fit the regression of next month's return on D/P, using only months whose next return you have already seen
- forecast next month's return with today's D/P
- the benchmark: the historical average return up to $t$, which is what you would use with no signal

The loop below does this every month from 1950. Read it before you run it.

In [ ]:
y = C['ex'].shift(-1)                      # next month's excess return, lined up with this month's D/P
rows = []
for t in C.loc['1950':].index[:-1]:
    known = pd.DataFrame({'y': y, 'dp': C['dp']}).loc['1927':t].iloc[:-1].dropna()   # pairs whose return is known at t
    b, a = np.polyfit(known['dp'], known['y'], 1)          # slope and intercept, from the past only
    rows.append({'date': t,
                 'r':        y.loc[t],                     # what happened next month
                 'forecast': a + b * C['dp'].loc[t],       # what D/P said
                 'average':  known['y'].mean()})           # what the historical average said
P = pd.DataFrame(rows).set_index('date')
P.head()

**Your turn.** The out-of-sample $R^2$ compares the two forecasts' squared errors:

$$R^2_{\text{OOS}} = 1 - \frac{\sum (r - \text{forecast})^2}{\sum (r - \text{average})^2}$$

Positive: D/P beat the historical average. Negative: you would have done better without it. Call it `r2_oos`.

In [ ]:
# === YOUR TURN ===

r2_oos = ____

In [ ]:
#@title 🔒 Check: out of sample
r2 = lambda Q: 1 - ((Q['r'] - Q['forecast']) ** 2).sum() / ((Q['r'] - Q['average']) ** 2).sum()

try:
    print(f"yours: {r2_oos:.2%}")
except Exception as e:
    print("yours: not built yet:", type(e).__name__)

print(f"out of sample R2, 1950-2024: {r2(P):.2%}")
for a, b in [('1950', '1979'), ('1980', '1999'), ('2000', '2024')]:
    print(f"   {a}-{b}: {r2(P.loc[a:b]):.2%}")

gain = ((P['r'] - P['average']) ** 2 - (P['r'] - P['forecast']) ** 2).cumsum()   # rising when D/P forecasts better
gain.plot(title='cumulative gain from D/P over the historical average'); plt.axhline(0, color='k', lw=0.5); plt.show()

- When did D/P stop working? Look at the plot.
- In sample, D/P explained 0.3% of next month's return. Out of sample it explains less than nothing. How can both be true?
- **Bring back to the slides:** the out-of-sample $R^2$, and when the plot turns.

---

## 4. The timing strategy

The formula from *Capital Allocation I* with a forecast in it: weight $= \text{forecast} / (\gamma \sigma^{2})$. We hold
$\sigma$ fixed at its value before 1950, so all the timing comes from the forecast.

In [ ]:
sigma2 = C['ex'].loc[:'1949'].var()        # the market's monthly variance before 1950, held fixed
gamma  = 3
weight = P['forecast'] / (gamma * sigma2)  # the weight on the market each month: negative means short
timed  = weight * P['r']                   # this month's excess return on that weight

print(f"timing: Sharpe {sharpe(timed):.2f}    buy and hold: Sharpe {sharpe(P['r']):.2f}")
print(f"average weight {weight.mean():.2f}, short in {np.mean(weight < 0):.0%} of months\n")
for a, b in [('1950', '1979'), ('1980', '1999'), ('2000', '2024')]:
    print(f"{a}-{b}: timing {sharpe(timed.loc[a:b]):.2f}, buy and hold {sharpe(P['r'].loc[a:b]):.2f}")

weight.plot(title='weight on the market'); plt.axhline(0, color='k', lw=0.5); plt.show()

- Would changing `gamma` change the Sharpe ratio? Try it.
- In which period did timing win, and in which did it lose badly? Why then?

---

## 5. Timing the six factors: 24 tries

Four simple signals for each factor: its own return last month, over the last 12 months, three to five years ago, and its
own volatility. Run the forecasting regression on 1980–2002 and again on 2003–2026. A real signal should work in both.

In [ ]:
d   = pd.read_parquet(f"{BASE}/ff6_daily_1963.parquet")                 # daily factors, for each factor's volatility
own_vol = d[F.columns].groupby(d.index + pd.offsets.MonthEnd(0)).apply(lambda x: np.sqrt(((x - x.mean()) ** 2).sum()))  # each month's volatility

rows = []
for f in F.columns:
    r = F[f]
    signals = {'last month':     r,
               'last 12 months': r.rolling(12).sum(),
               '3-5 years ago':  r.shift(12).rolling(48).sum(),
               'own volatility': own_vol[f].reindex(r.index)}
    for name, s in signals.items():
        data = pd.DataFrame({'y': r.shift(-1), 'x': s}).dropna()       # next month's return on this month's signal
        row = {'factor': f, 'signal': name}
        for half, sl in [('t, 1980-2002', slice('1980', '2002')), ('t, 2003-2026', slice('2003', '2026'))]:
            q = data.loc[sl]
            row[half] = sm.OLS(q['y'], sm.add_constant(q['x'])).fit().tvalues['x']
        rows.append(row)
T = pd.DataFrame(rows)
print(T.round(2).sort_values('t, 1980-2002', ascending=False).to_string(index=False))

plt.scatter(T['t, 1980-2002'], T['t, 2003-2026'])
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plt.xlabel('t, 1980-2002'); plt.ylabel('t, 2003-2026'); plt.show()

- How many of the 24 have $|t| > 2$ in the first half? How many would you expect by luck alone?
- Take the five best from the first half. How many still work in the second?
- Which signal works for several factors in both halves? It has a name: factor momentum.
- **Bring back to the slides.**

---

## ⏸ Back to the slides

The challenge below is homework.

---

## 🎯 Challenge: start later

*Homework — due before the next class. It should take ten minutes.*

Section 3 started forecasting in 1950. Suppose you had only started in **1980**. Rerun the loop from section 3 with
`C.loc['1980':]` instead of `C.loc['1950':]`, and everything else the same. Report the out-of-sample $R^2$.

Then, in at most three sentences: would you trade D/P on this evidence? Why or why not?

> **📌 Required variable names:**
> ```python
> oos_r2_1980 = ____   # out-of-sample R2 with forecasts from 1980, a decimal (-0.01 means -1%)
> MEMO = """...""" # at most three sentences
> ```

In [ ]:
# Your work here


oos_r2_1980 = ____

print(f"out of sample R2 from 1980: {oos_r2_1980:.2%}")

In [ ]:
MEMO = """
Write your answer here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ['oos_r2_1980', 'MEMO']
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L15_ConditionalStrategiesII_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")